# TP N°4 : Indexation sémantique avec LDA

**Objectif :** Comprendre le fonctionnement du modèle LDA pour la modélisation de sujets (topic modeling), prétraiter un corpus, appliquer un modèle LDA, indexer des documents et évaluer la qualité de l'indexation thématique.

## II. Installation des bibliothèques

In [ ]:
!pip install gensim nltk matplotlib pyLDAvis

## III. Préparation du corpus
### 1. Téléchargement du corpus
Roman : *Crime and Punishment* de Fyodor Dostoyevsky

In [ ]:
import urllib.request

url = "https://www.gutenberg.org/cache/epub/2554/pg2554.txt"
response = urllib.request.urlopen(url)
raw_text = response.read().decode("utf-8")

# Suppression de l'en-tête et du pied de page du Projet Gutenberg
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"
start = raw_text.find(start_marker)
end = raw_text.find(end_marker)
if start != -1:
    raw_text = raw_text[raw_text.index("\n", start) + 1 : end]

print(f"Longueur du texte : {len(raw_text)} caractères")
print(raw_text[:500])

### 2. Prétraitement
- Tokenisation en phrases puis en mots
- Suppression de la ponctuation, des majuscules, des mots vides (stopwords)
- Lemmatisation

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

# Tokenisation en phrases
sentences = sent_tokenize(raw_text)
print(f"Nombre de phrases : {len(sentences)}")

# Initialisation
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(sentence):
    """Tokenise, nettoie et lemmatise une phrase."""
    # Mise en minuscules
    sentence = sentence.lower()
    # Tokenisation en mots
    tokens = word_tokenize(sentence)
    # Suppression ponctuation, mots vides, mots très courts
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word.isalpha() and word not in stop_words and len(word) > 2
    ]
    return tokens

# Application du prétraitement sur chaque phrase
processed_docs = [preprocess(sent) for sent in sentences]

# Suppression des phrases vides après prétraitement
processed_docs = [doc for doc in processed_docs if len(doc) > 0]

print(f"Nombre de documents (phrases) après nettoyage : {len(processed_docs)}")
print("Exemple de document prétraité :")
print(processed_docs[0])

## IV. Construction du modèle LDA
### 1. Dictionnaire et Bag of Words

In [ ]:
from gensim import corpora

# Création du dictionnaire
dictionary = corpora.Dictionary(processed_docs)

# Filtrage : suppression des mots trop rares ou trop fréquents
dictionary.filter_extremes(no_below=5, no_above=0.5)

print(f"Taille du dictionnaire : {len(dictionary)} termes")

# Conversion en Bag of Words
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

print(f"Nombre de documents dans le corpus BoW : {len(corpus)}")
print("Exemple BoW du premier document :")
print(corpus[0])

### 2. Entraînement du modèle LDA (10 topics)

In [ ]:
from gensim.models import LdaModel

# Entraînement du modèle LDA avec 10 topics
num_topics = 10
model_lda = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=10,
    alpha="auto",
    per_word_topics=True,
)

# Affichage des mots caractéristiques de chaque sujet
print("=" * 60)
print(f"Mots caractéristiques pour {num_topics} topics :")
print("=" * 60)
for idx, topic in model_lda.print_topics(-1, num_words=10):
    print(f"\nTopic {idx} : {topic}")

### 3. Questions théoriques

**Qu'est-ce qu'un « topic » dans le contexte du LDA ?**

Un *topic* (sujet) dans LDA est une **distribution de probabilités sur l'ensemble du vocabulaire**. Chaque topic attribue une probabilité élevée à un groupe de mots qui tendent à apparaître ensemble dans les documents. Par exemple, un topic peut avoir des probabilités élevées pour les mots « murder », « blood », « crime », ce qui suggère un thème lié à la violence/crime.

LDA suppose que chaque document est un mélange de topics, et chaque mot du document est généré en choisissant d'abord un topic (selon la distribution du document), puis en choisissant un mot (selon la distribution du topic).

---

**Quel est le rôle du dictionnaire (`corpora.Dictionary`) et de la représentation Bag-of-Words ?**

- Le **dictionnaire** (`corpora.Dictionary`) associe chaque mot unique du corpus à un identifiant numérique. Il sert de mapping entre les tokens textuels et leurs indices numériques.
- La représentation **Bag-of-Words** (BoW) convertit chaque document en une liste de paires `(id_mot, fréquence)`. Elle ignore l'ordre des mots et ne conserve que les fréquences d'apparition. C'est le format d'entrée requis par le modèle LDA de Gensim.

### 4. Variation du nombre de topics (5, 15, 20)

In [ ]:
from gensim.models import CoherenceModel

results = {}

for n_topics in [5, 10, 15, 20]:
    lda_temp = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=n_topics,
        random_state=42,
        passes=10,
        alpha="auto",
        per_word_topics=True,
    )

    # Calcul de la cohérence (c_v)
    coherence_model = CoherenceModel(
        model=lda_temp,
        texts=processed_docs,
        dictionary=dictionary,
        coherence="c_v",
    )
    coherence_score = coherence_model.get_coherence()
    results[n_topics] = coherence_score

    print(f"\n{'=' * 60}")
    print(f"num_topics = {n_topics}  |  Cohérence (c_v) = {coherence_score:.4f}")
    print("=" * 60)
    for idx, topic in lda_temp.print_topics(-1, num_words=8):
        print(f"  Topic {idx}: {topic}")

In [ ]:
import matplotlib.pyplot as plt

# Graphique de cohérence en fonction du nombre de topics
plt.figure(figsize=(8, 5))
plt.plot(list(results.keys()), list(results.values()), marker="o", linewidth=2)
plt.xlabel("Nombre de topics")
plt.ylabel("Score de cohérence (c_v)")
plt.title("Cohérence du modèle LDA selon le nombre de topics")
plt.xticks(list(results.keys()))
plt.grid(True)
plt.show()

### Observations sur la variation du nombre de topics

- **5 topics** : Les thèmes sont très larges et mélangés. Plusieurs sujets distincts du roman se retrouvent fusionnés dans un même topic. La granularité est insuffisante pour distinguer les différents arcs narratifs.

- **10 topics** : C'est généralement un bon compromis. Les topics commencent à capturer des thèmes cohérents : crime/meurtre, famille, pauvreté, enquête policière, rédemption, etc.

- **15 topics** : On obtient des topics plus spécifiques, mais certains commencent à se chevaucher ou à devenir redondants (deux topics sur des thèmes très similaires).

- **20 topics** : Trop de topics entraîne une fragmentation excessive. Des topics deviennent difficiles à interpréter car ils contiennent des mots peu cohérents entre eux. Le modèle « sur-segmente » les thèmes.

## V. Visualisation et interprétation

In [ ]:
import pyLDAvis.gensim_models
import pyLDAvis

pyLDAvis.enable_notebook()
vis_data = pyLDAvis.gensim_models.prepare(model_lda, corpus, dictionary)
vis_data

In [ ]:
# Sauvegarde de la visualisation en fichier HTML
pyLDAvis.save_html(vis_data, "lda_visualization.html")
print("Visualisation sauvegardée dans 'lda_visualization.html'")

### Interprétation de la visualisation pyLDAvis

**Que représentent les cercles affichés ?**

Chaque cercle représente un **topic**. Leur position est déterminée par une réduction de dimensionnalité (PCA) appliquée aux distributions de mots des topics.

- **La taille** d'un cercle représente la **prévalence** (fréquence) du topic dans le corpus. Un cercle plus grand signifie que ce topic explique une plus grande proportion des mots du corpus.

- **L'éloignement** (distance entre cercles) représente la **dissimilarité sémantique** entre les topics. Deux cercles éloignés signifient que les topics traitent de thèmes très différents. Deux cercles proches ou qui se chevauchent indiquent des thèmes similaires ou redondants.

---

### Interprétation de deux topics bien séparés

*(Les mots exacts dépendent de l'exécution — voici une interprétation typique basée sur le roman)*

**Topic A — « Le Crime et la Culpabilité »**
Mots dominants typiques : *murder, blood, old, woman, axe, door, room, fear, hand, body*
Ce topic capte les scènes du meurtre de la prêteuse sur gages et la culpabilité qui s'ensuit. Il couvre les passages où Raskolnikov prépare et exécute son crime.

**Topic B — « La Famille et la Pauvreté »**
Mots dominants typiques : *mother, sister, money, family, letter, poor, love, marry, daughter, father*
Ce topic capture les relations familiales : la correspondance avec sa mère et sa sœur Dounia, les difficultés financières, et le sacrifice de la famille.

---

### Passage du roman associé à un thème

Pour le topic « Le Crime et la Culpabilité », un passage illustratif se trouve dans la **Partie I, Chapitre VII** :

> *"He took the axe right out, swung it with both arms, scarcely conscious of himself, and almost without effort, almost mechanically, brought the blunt side down on her head."*

Ce passage décrit directement le meurtre et correspond aux mots-clés détectés par ce topic.

### Indexation d'un document par sa distribution thématique

In [ ]:
# Affichage de la distribution thématique pour quelques documents
print("Distribution thématique des 5 premiers documents :")
print("=" * 60)
for i in range(5):
    topic_dist = model_lda.get_document_topics(corpus[i])
    print(f"\nDocument {i} : {' '.join(processed_docs[i][:10])}...")
    for topic_id, prob in sorted(topic_dist, key=lambda x: -x[1]):
        print(f"  Topic {topic_id}: {prob:.4f}")

In [ ]:
# Recherche des documents dominés par un topic donné
target_topic = 0  # Changer ce numéro pour explorer d'autres topics

print(f"Documents les plus associés au Topic {target_topic} :")
print("=" * 60)

doc_topic_scores = []
for i, bow in enumerate(corpus):
    topic_dist = model_lda.get_document_topics(bow)
    for tid, prob in topic_dist:
        if tid == target_topic:
            doc_topic_scores.append((i, prob))
            break

# Tri par score décroissant et affichage du top 5
doc_topic_scores.sort(key=lambda x: -x[1])
for rank, (doc_idx, score) in enumerate(doc_topic_scores[:5]):
    print(f"\n#{rank + 1} (score={score:.4f}) — Phrase originale :")
    print(f"  {sentences[doc_idx][:200]}...")